# 03 · Extreme Value Analysis and IDF Curves

**Author:** Salvador Navas  
**Basin:** Río Besaya — Cantabria

Starting from annual daily rainfall maxima, this notebook:
- Fits GEV and Gumbel distributions to annual block maxima (BM)
- Estimates return levels for T = 2, 5, 10, 25, 50, 100, 500 years
- Constructs IDF curves for the basin using duration-scaling relationships
- Exports parameters and quantiles for Notebook 04

**Methodological note.** The record has 43 annual maxima. Return levels beyond
about 100 years are extrapolations with high uncertainty; the 500-year value is
kept for workflow continuity, not as a calibrated design recommendation.


In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pyhydra.climate.time_series.extremes import (
    extract_block_maxima,
    fit_gev, fit_gpd,
    return_levels,
    return_level_ci,
    plot_return_levels,
    plot_diagnostic,
)

# Resolve repo root: Jupyter sets cwd to the notebook dir;
# in Docker/script contexts cwd=/workspace (one level up)
_cwd = Path.cwd()
_candidates = [Path('/workspace'), _cwd, *_cwd.parents]
REPO_ROOT = next(
    (p for p in _candidates if (p / 'data' / 'pilot_cases' / 'los_corrales_buelna').exists()),
    _cwd.parent.parent.parent if _cwd.name == 'los_corrales_buelna' else _cwd,
)
DATA_ROOT = REPO_ROOT / 'data' / 'pilot_cases' / 'los_corrales_buelna'
PROC_DIR  = DATA_ROOT / 'processed'

RETURN_PERIODS = [2, 5, 10, 25, 50, 100, 500]
DURATIONS_H    = [1, 2, 3, 6, 12, 24]   # horas — 24h = datos diarios

## 1. Load daily AMR and extract annual maxima

In [2]:
# ── Option A: load pre-computed annual maxima (.dat per station) ────────────
DAT_FILES = sorted((DATA_ROOT / 'gauges' / 'rainfall').glob('Maximos_anual_*.dat'))
print(f'Ficheros .dat encontrados: {[f.stem for f in DAT_FILES]}')

ann_max_series = {}
for dat in DAT_FILES:
    station = dat.stem.replace('Maximos_anual_', '')
    values  = np.loadtxt(dat)          # columna simple de valores en mm/día
    n       = len(values)
    # Assign years (last year of record is approx. 2012)
    years   = range(2012 - n + 1, 2013)
    idx     = pd.to_datetime([f'{y}-12-31' for y in years])
    ann_max_series[station] = pd.Series(values, index=idx, name=station)

ann_max_df = pd.DataFrame(ann_max_series)
print(f'Máximos anuales por estación: {ann_max_df.shape}')
print(ann_max_df.describe().round(1))

# Spatial mean for basin frequency analysis
annual_max = ann_max_df.mean(axis=1).dropna()
annual_max.name = 'PMA_Besaya'

# ── Option B: compute from interpolated daily AMR (requires Notebook 02) ───
# pma = pd.read_csv(PROC_DIR / 'pma_idw_daily.csv', index_col=0, parse_dates=True)
# annual_max = extract_block_maxima(pma.mean(axis=1), freq='YE').dropna()
# ──────────────────────────────────────────────────────────────────────────────

print(f'\nMáximos anuales de la cuenca: {len(annual_max)} años')
print(f'  min={annual_max.min():.1f}  media={annual_max.mean():.1f}  max={annual_max.max():.1f} mm/día')
annual_max.tail()

Ficheros .dat encontrados: ['Maximos_anual_1140E', 'Maximos_anual_1144', 'Maximos_anual_1151', 'Maximos_anual_1153E', 'Maximos_anual_144']
Máximos anuales por estación: (43, 5)
       1140E  1144   1151  1153E   144
count   43.0  43.0   43.0   43.0  43.0
mean    47.7  48.9   54.8   51.8  48.9
std     11.5  12.5   18.7   14.3  12.5
min     30.4  29.7   27.7   28.6  29.7
25%     40.2  40.2   42.5   42.0  40.2
50%     46.9  45.3   52.1   47.3  45.3
75%     52.5  55.4   62.0   63.1  55.4
max     82.8  89.5  122.2   79.3  89.5

Máximos anuales de la cuenca: 43 años
  min=34.7  media=50.4  max=78.8 mm/día


## 2. GEV and Gumbel fitting

In [3]:
# GEV (MLE with bias correction)
params_gev = fit_gev(annual_max.values, method='mle')
print('GEV    — mu={mu:.2f}  sigma={sigma:.2f}  xi={xi:.3f}'.format(**params_gev))

# Gumbel = GEV special case with xi = 0 (fitted via scipy gumbel_r)
from scipy.stats import gumbel_r as _gumbel
_loc, _sc = _gumbel.fit(annual_max.values)
params_gumbel = {'mu': float(_loc), 'sigma': float(_sc), 'xi': 0.0}
print('Gumbel — mu={mu:.2f}  sigma={sigma:.2f}  xi=0 (fixed)'.format(**params_gumbel))


GEV    — mu=45.31  sigma=8.36  xi=0.032
Gumbel — mu=45.46  sigma=8.47  xi=0 (fixed)


In [4]:
# Visual diagnostic
fig = plot_diagnostic(annual_max.values, params_gev, dist='gev')
plt.savefig(PROC_DIR / 'gev_diagnostic.png', dpi=150)
plt.show()


## 3. Return levels and confidence bands (95 %)

In [5]:
T_plot = np.logspace(np.log10(1.1), np.log10(1000), 200)

rl_gev    = return_levels(params_gev,    T_plot, dist='gev')
rl_gumbel = return_levels(params_gumbel, T_plot, dist='gev')

# Bootstrap confidence intervals — one call per return period
ci_gev = {}
for T in RETURN_PERIODS:
    ci_gev[T] = return_level_ci(annual_max.values, T,
                                 dist='gev', n_bootstrap=200, ci=0.95)
# ci_gev[T] = (estimate, lower_ci, upper_ci)

fig = plot_return_levels(
    annual_max.values, params_gev,
    dist='gev', ci=0.95
)
plt.savefig(PROC_DIR / 'return_levels.png', dpi=150)
plt.show()


In [6]:
# Quantile table for T = RETURN_PERIODS
rl_table = pd.DataFrame({
    'T_years':   RETURN_PERIODS,
    'GEV_mm':    [return_levels(params_gev,    [T], dist='gev').iloc[0]
                  for T in RETURN_PERIODS],
    'Gumbel_mm': [return_levels(params_gumbel, [T], dist='gev').iloc[0]
                  for T in RETURN_PERIODS],
    'CI_low_mm':  [ci_gev[T][1] for T in RETURN_PERIODS],
    'CI_high_mm': [ci_gev[T][2] for T in RETURN_PERIODS],
})
rl_table = rl_table.round(1)
print(rl_table.to_string(index=False))
rl_table.to_csv(PROC_DIR / 'return_levels_24h.csv', index=False)


 T_years  GEV_mm  Gumbel_mm  CI_low_mm  CI_high_mm
       2    48.4       48.6       44.4        52.0
       5    58.2       58.2       53.3        61.7
      10    64.8       64.5       58.7        69.6
      25    73.5       72.6       64.3        84.3
      50    80.1       78.5       68.2        97.6
     100    86.8       84.4       72.1       115.2
     500   102.8       98.1       79.5       187.7


## 4. IDF curves (Intensity–Duration–Frequency)

Temporal disaggregation relationships are used to convert P24h to sub-daily values.  
Témez scaling relationship (MOPU, 1992) adapted for the Cantabrian (Atlantic) climate:
$$P_d = P_{24h} \cdot \left(\frac{d}{24}\right)^{0.28}$$


In [7]:
TEMEZ_EXPONENT = 0.28   # exponent for Cantabria / Atlantic climate

idf_records = []
for T in RETURN_PERIODS:
    p24 = return_levels(params_gev, [T], dist='gev').iloc[0]
    for d in DURATIONS_H:
        pd_mm  = p24 * (d / 24) ** TEMEZ_EXPONENT
        i_mmh  = pd_mm / d
        idf_records.append({'T_years': T, 'duration_h': d,
                            'depth_mm': round(pd_mm, 1),
                            'intensity_mmh': round(i_mmh, 2)})

idf = pd.DataFrame(idf_records)
idf_pivot = idf.pivot(index='duration_h', columns='T_years', values='depth_mm')
print('IDF — Profundidades de lluvia (mm):')
print(idf_pivot.to_string())
idf.to_csv(PROC_DIR / 'idf_curves.csv', index=False)

IDF — Profundidades de lluvia (mm):
T_years      2     5     10    25    50    100    500
duration_h                                           
1           19.9  23.9  26.6  30.2  32.9  35.6   42.2
2           24.1  29.0  32.3  36.6  39.9  43.3   51.3
3           27.0  32.5  36.2  41.1  44.7  48.5   57.4
6           32.8  39.5  44.0  49.8  54.3  58.9   69.7
12          39.9  47.9  53.4  60.5  65.9  71.5   84.7
24          48.4  58.2  64.8  73.5  80.1  86.8  102.8


In [8]:
idf_int = idf.pivot(index='duration_h', columns='T_years', values='intensity_mmh')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.plasma(np.linspace(0, 0.85, len(RETURN_PERIODS)))
for (T, col), c in zip(idf_pivot.items(), colors):
    axes[0].plot(idf_pivot.index, col, 'o-', label=f'T={T} años', color=c)
axes[0].set(xlabel='Duración (h)', ylabel='Profundidad (mm)',
            title='IDF — Profundidad', xscale='log', yscale='log')
axes[0].legend(fontsize=8)
axes[0].grid(True, which='both', alpha=0.3)

for (T, col), c in zip(idf_int.items(), colors):
    axes[1].plot(idf_int.index, col, 'o-', label=f'T={T} años', color=c)
axes[1].set(xlabel='Duración (h)', ylabel='Intensidad (mm/h)',
            title='IDF — Intensidad', xscale='log', yscale='log')
axes[1].legend(fontsize=8)
axes[1].grid(True, which='both', alpha=0.3)

plt.suptitle('Curvas IDF — Cuenca del Besaya', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROC_DIR / 'idf_curves.png', dpi=150)
plt.show()

## 5. Seasonal analysis of extreme events

In [9]:
# Month in which each annual maximum occurs
annual_max_df = pd.DataFrame({'pma_mm': annual_max.values}, index=annual_max.index)
annual_max_df['month'] = annual_max_df.index.month

monthly_counts = annual_max_df['month'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(monthly_counts.index,  monthly_counts.values, color='steelblue', edgecolor='k')
ax.set(xlabel='Mes', ylabel='Frecuencia de máximos anuales',
       title='Estacionalidad de los máximos anuales — PMA Besaya',
       xticks=range(1, 13),
       xticklabels=['Ene','Feb','Mar','Abr','May','Jun',
                    'Jul','Ago','Sep','Oct','Nov','Dic'])
plt.tight_layout()
plt.savefig(PROC_DIR / 'seasonality_maxima.png', dpi=150)
plt.show()

## 6. Summary of exported parameters

In [10]:
import json

summary = {
    'n_years':      int(len(annual_max)),
    'period_start': str(annual_max.index[0].date()),
    'period_end':   str(annual_max.index[-1].date()),
    'gev_params':   {k: round(float(v), 4) for k, v in params_gev.items()},
    'gumbel_params': {k: round(float(v), 4) for k, v in params_gumbel.items()},
    'return_levels_24h': {str(T): round(float(return_levels(params_gev, [T], dist='gev').iloc[0]), 1)
                          for T in RETURN_PERIODS},
    'idf_temez_exponent': TEMEZ_EXPONENT,
}

with open(PROC_DIR / 'extreme_value_params.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Exportado: extreme_value_params.json')
print(json.dumps(summary, indent=2))

Exportado: extreme_value_params.json
{
  "n_years": 43,
  "period_start": "1970-12-31",
  "period_end": "2012-12-31",
  "gev_params": {
    "mu": 45.3086,
    "sigma": 8.3649,
    "xi": 0.032
  },
  "gumbel_params": {
    "mu": 45.4555,
    "sigma": 8.4721,
    "xi": 0.0
  },
  "return_levels_24h": {
    "2": 48.4,
    "5": 58.2,
    "10": 64.8,
    "25": 73.5,
    "50": 80.1,
    "100": 86.8,
    "500": 102.8
  },
  "idf_temez_exponent": 0.28
}
